In [6]:
import argparse
from xuance.common import get_configs
from xuance.environment import make_envs
import numpy as np
import importlib
import sattelite_env as sat_env
from xuance.environment import REGISTRY_ENV
from xuance.environment import make_envs
from xuance.torch.agents import DDPG_Agent, PPOCLIP_Agent, DreamerV3Agent

configs_dict = get_configs(file_dir="configs/DDPG_satellite_env.yaml")
configs = argparse.Namespace(**configs_dict)

importlib.reload(sat_env)
REGISTRY_ENV[configs.env_name] = sat_env.SatelliteMECEnvironment
envs = make_envs(configs)
obs, info = envs.reset()
print('envs ready:', type(envs).__name__)
print('obs shape:', obs.shape if isinstance(obs, np.ndarray) else type(obs))
print('info len:', len(info) if isinstance(info, list) else type(info))
envs.close()


envs = make_envs(configs)  # Make parallel environments.
Agent = DDPG_Agent(config=configs, envs=envs)  # Create a DDPG agent from XuanCe.
Agent.train(configs.running_steps // configs.parallels)  # Train the model for numerous steps.
Agent.save_model("final_train_model.pth")  # Save the model to model_dir.
Agent.finish()  # Finish the training.

2026-02-28 02:30:45,426 - sattelite_env - INFO - Initializing SystemParameters with seed=1
2026-02-28 02:30:45,426 - sattelite_env - INFO - Initialized Satellite-MEC Environment with 50 MGUs
2026-02-28 02:30:45,427 - sattelite_env - INFO - Initializing SystemParameters with seed=2
2026-02-28 02:30:45,427 - sattelite_env - INFO - Initialized Satellite-MEC Environment with 50 MGUs
2026-02-28 02:30:45,427 - sattelite_env - INFO - Initializing SystemParameters with seed=3
2026-02-28 02:30:45,428 - sattelite_env - INFO - Initialized Satellite-MEC Environment with 50 MGUs
2026-02-28 02:30:45,429 - sattelite_env - INFO - Initializing SystemParameters with seed=3
2026-02-28 02:30:45,429 - sattelite_env - INFO - Initialized Satellite-MEC Environment with 50 MGUs
2026-02-28 02:30:45,430 - sattelite_env - INFO - Initializing SystemParameters with seed=4
2026-02-28 02:30:45,430 - sattelite_env - INFO - Initialized Satellite-MEC Environment with 50 MGUs
2026-02-28 02:30:45,430 - sattelite_env - INF

envs ready: DummyVecEnv
obs shape: (3, 154)
info len: 3


100%|██████████| 5000/5000 [01:04<00:00, 77.04it/s]


In [7]:

configs_dict = get_configs(file_dir="configs/PPO_satellite_env.yaml")
configs = argparse.Namespace(**configs_dict)
importlib.reload(sat_env)
REGISTRY_ENV[configs.env_name] = sat_env.SatelliteMECEnvironment
envs = make_envs(configs)

Agent = PPOCLIP_Agent(config=configs, envs=envs)  # Create a PPOCLIP agent from XuanCe.
Agent.train(configs.running_steps // configs.parallels)  # Train the model for numerous steps.
Agent.save_model("final_train_model.pth")  # Save the model to model_dir.
Agent.finish()  # Finish the training.

2026-02-28 02:31:50,357 - sattelite_env - INFO - Initializing SystemParameters with seed=1
2026-02-28 02:31:50,358 - sattelite_env - INFO - Initialized Satellite-MEC Environment with 50 MGUs
2026-02-28 02:31:50,359 - sattelite_env - INFO - Initializing SystemParameters with seed=2
2026-02-28 02:31:50,359 - sattelite_env - INFO - Initialized Satellite-MEC Environment with 50 MGUs
2026-02-28 02:31:50,360 - sattelite_env - INFO - Initializing SystemParameters with seed=3
2026-02-28 02:31:50,360 - sattelite_env - INFO - Initialized Satellite-MEC Environment with 50 MGUs
100%|██████████| 5000/5000 [00:55<00:00, 89.55it/s] 


In [4]:
# DreamerV3 training with custom Satellite env (standalone cell)
import argparse
import importlib
import sattelite_env as sat_env
from xuance.common import get_configs
from xuance.environment import REGISTRY_ENV, make_envs
from xuance.torch.agents import DreamerV3Agent

# Workaround for xuance==1.4.1 where DreamerV3Agent references self.envs
if not hasattr(DreamerV3Agent, "envs"):
    DreamerV3Agent.envs = property(lambda self: self.train_envs)

configs_dict = get_configs(file_dir="configs/Dreamer_satellite_env.yaml")
configs = argparse.Namespace(**configs_dict)

importlib.reload(sat_env)
REGISTRY_ENV[configs.env_name] = sat_env.SatelliteMECEnvironment
envs = make_envs(configs)

# quick smoke run first
Agent = DreamerV3Agent(config=configs, envs=envs)
Agent.train(configs.running_steps // configs.parallels)
Agent.save_model("dreamerv3_smoke_model.pth")
Agent.finish()

2026-02-28 02:15:31,246 - sattelite_env - INFO - Initializing SystemParameters with seed=1
2026-02-28 02:15:31,247 - sattelite_env - INFO - Initialized Satellite-MEC Environment with 50 MGUs
2026-02-28 02:15:31,247 - sattelite_env - INFO - Initializing SystemParameters with seed=2
2026-02-28 02:15:31,247 - sattelite_env - INFO - Initialized Satellite-MEC Environment with 50 MGUs
100%|██████████| 5000/5000 [14:32<00:00,  5.73it/s]

DreamerV3 train done, steps=200


In [2]:
# DiffPPO training with custom Satellite env (standalone cell)
import argparse
import importlib
import sattelite_env as sat_env

from xuance.common import get_configs
from xuance.environment import REGISTRY_ENV, make_envs
from custemize_RL import DiffusionPPOAgent  # class custom bạn đã viết

# load config DiffPPO
configs_dict = get_configs(file_dir="configs/DiffPPO_config.yaml")
configs = argparse.Namespace(**configs_dict)

# register custom env
importlib.reload(sat_env)
REGISTRY_ENV[configs.env_name] = sat_env.SatelliteMECEnvironment

# build envs
envs = make_envs(configs)

# train
Agent = DiffusionPPOAgent(config=configs, envs=envs)
Agent.train(configs.running_steps // configs.parallels)
Agent.save_model("diffppo_smoke_model.pth")
Agent.finish()
envs.close()

2026-02-28 13:19:47,766 - sattelite_env - INFO - Initializing SystemParameters with seed=1
2026-02-28 13:19:47,767 - sattelite_env - INFO - Initialized Satellite-MEC Environment with 50 MGUs
2026-02-28 13:19:47,768 - sattelite_env - INFO - Initializing SystemParameters with seed=2
2026-02-28 13:19:47,768 - sattelite_env - INFO - Initialized Satellite-MEC Environment with 50 MGUs
100%|██████████| 5/5 [00:00<00:00, 124.12it/s]


In [7]:
!tensorboard --logdir=logs/ --port=6006


/home/n2tp/miniconda3/envs/xuance/lib/python3.10/site-packages/tensorboard/default.py:30: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


TensorFlow installation not found - running with reduced feature set.

NOTE: Using experimental fast data loading logic. To disable, pass
    "--load_fast=false" and report issues on GitHub. More details:
    https://github.com/tensorflow/tensorboard/issues/4784

E0227 12:13:38.392635 127841150338880 program.py:300] TensorBoard could not bind to port 6006, it was already in use
ERROR: TensorBoard could not bind to port 6006, it was already in use
